## Using CNN for Cat vs Dog Classification task

In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

dataset_path = r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset"

cats = os.listdir(dataset_path + r"\Cat")
dogs = os.listdir(dataset_path + r"\Dog")

train_cats, test_cats = train_test_split(cats, test_size=0.3, random_state=42)
val_cats, test_cats = train_test_split(test_cats, test_size=0.5)

train_dogs, test_dogs = train_test_split(dogs, test_size=0.3, random_state=42)
val_dogs, test_dogs = train_test_split(test_dogs, test_size=0.5)

In [2]:
from PIL import Image

img_path = dataset_path + "/Dog/" + train_dogs[5]
img = Image.open(img_path)
img.show()  # opens in your system image viewer

In [3]:
base_dir = r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\dataset_split"

# Create folder structure
for split in ["train", "val", "test"]:
    for category in ["Cat", "Dog"]:
        os.makedirs(os.path.join(base_dir, split, category), exist_ok=True)

In [4]:
# Function to copy images
def copy_images(file_list, source_folder, target_folder):
    for filename in file_list:
        shutil.copy(
            os.path.join(source_folder, filename),
            os.path.join(target_folder, filename)
        )

# Copy Cat images
copy_images(train_cats, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Cat", base_dir+"/train/Cat")
copy_images(val_cats, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Cat", base_dir+"/val/Cat")
copy_images(test_cats, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Cat", base_dir+"/test/Cat")

# Copy Dog images
copy_images(train_dogs, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Dog", base_dir+"/train/Dog")
copy_images(val_dogs, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Dog", base_dir+"/val/Dog")
copy_images(test_dogs, r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\cats_dogs_dataset/Dog", base_dir+"/test/Dog")

In [5]:
train_cats_path = os.path.join(base_dir, "train/Cat")
print("Train Cat images:", len(os.listdir(train_cats_path)))
print("Val Cat images:", len(os.listdir(os.path.join(base_dir,"val/Cat"))))
print("Test Cat images:", len(os.listdir(os.path.join(base_dir,"test/Cat"))))
img_list = os.listdir(train_cats_path)
type(img_list)
type(img_list[0])

Train Cat images: 8749
Val Cat images: 3266
Test Cat images: 3277


str

In [6]:
train_dogs_path = os.path.join(base_dir, "train/Dog")
print("Train Dog images:", len(os.listdir(train_dogs_path)))
print("Val Dog images:", len(os.listdir(os.path.join(base_dir,"val/Dog"))))
print("Test Dog images:", len(os.listdir(os.path.join(base_dir,"test/Dog"))))
img_list = os.listdir(train_dogs_path)
type(img_list)
type(img_list[0])

Train Dog images: 8749
Val Dog images: 3287
Test Dog images: 3288


str

In [7]:
import os
import numpy as np
from PIL import Image
from tensorflow.keras import models, layers

# Parameters
IMG_SIZE = (64, 64)  # Resize images
base_dir = r"C:\Users\GASCO\Desktop\Assignment Lab\Cats Dogs Classification\dataset_split"

def load_images_and_labels(folder):
    X = []
    y = []
    for label, category in enumerate(["Cat", "Dog"]):  # Cat=0, Dog=1
        category_folder = os.path.join(folder, category)
        for file in os.listdir(category_folder):
            try:
                img = Image.open(os.path.join(category_folder, file)).convert('RGB')
                img = img.resize(IMG_SIZE)
                img_array = np.array(img)
                X.append(img_array)
                y.append(label)
            except:
                pass
    return np.array(X), np.array(y)

# Load datasets
X_train, y_train = load_images_and_labels(os.path.join(base_dir, "train"))
X_val, y_val = load_images_and_labels(os.path.join(base_dir, "val"))
X_test, y_test = load_images_and_labels(os.path.join(base_dir, "test"))


print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

c:\Users\GASCO\anaconda3\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Train shape: (17498, 64, 64, 3) (17498,)
Val shape: (6553, 64, 64, 3) (6553,)
Test shape: (6565, 64, 64, 3) (6565,)


In [8]:
# Normalize images
X_train = X_train / 255.0
X_test = X_test / 255.0

In [10]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout , BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# CNN Model
model = Sequential([
    # First block
    Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Second block
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Third block
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    # Flatten and Dense layers
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,274,305 (4.86 MB)

 Trainable params: 1,273,857 (4.86 MB)

 Non-trainable params: 448 (1.75 KB)

In [11]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# --- Optional: Data augmentation ---
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)
datagen.fit(X_train)

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',   # what to watch
    patience=4,           # how many epochs to wait
    restore_best_weights=True
)

# --- Train the model ---
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=20,
    callbacks=[early_stop]
)


Epoch 1/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 85s 149ms/step - accuracy: 0.6087 - loss: 0.7318 - val_accuracy: 0.6417 - val_loss: 0.6259
Epoch 2/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 74s 135ms/step - accuracy: 0.6751 - loss: 0.6049 - val_accuracy: 0.6635 - val_loss: 0.6019
Epoch 3/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 82s 150ms/step - accuracy: 0.7124 - loss: 0.5638 - val_accuracy: 0.7479 - val_loss: 0.5154
Epoch 4/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 94s 173ms/step - accuracy: 0.7385 - loss: 0.5316 - val_accuracy: 0.6995 - val_loss: 0.5778
Epoch 5/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 109s 200ms/step - accuracy: 0.7615 - loss: 0.4977 - val_accuracy: 0.7420 - val_loss: 0.5284
Epoch 6/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 111s 203ms/step - accuracy: 0.7835 - loss: 0.4679 - val_accuracy: 0.7823 - val_loss: 0.4741
Epoch 7/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 117s 213ms/step - accuracy: 0.7968 - loss: 0.4448 - val_accuracy: 0.8203 - val_loss: 0.4048
Epoch 8/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 115s 211ms/step - accuracy: 0.8049 - los

In [12]:
# --- Evaluate model ---
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")


206/206 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.8813 - loss: 0.2772
Test Accuracy: 88.13%


In [ ]:
import tensorflow as tf
y_pred = model.predict (X_test)
y_prd_ex = 1 if tf.sigmoid(y_pred[430]) >= 0.5 else 0
print (f"Predition of {3660}th example is {y_prd_ex}")




118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
Predition of 3660th example is 1


In [ ]:
X_test[430]

array([[[0.17254902, 0.03921569, 0.00392157],
        [0.17647059, 0.03529412, 0.00392157],
        [0.18431373, 0.03137255, 0.00392157],
        ...,
        [0.69019608, 0.42352941, 0.13333333],
        [0.65490196, 0.38431373, 0.10588235],
        [0.67843137, 0.40392157, 0.14117647]],

       [[0.17254902, 0.03921569, 0.00392157],
        [0.17647059, 0.03529412, 0.00392157],
        [0.18431373, 0.03137255, 0.00392157],
        ...,
        [0.83529412, 0.57647059, 0.2627451 ],
        [0.84705882, 0.59215686, 0.2627451 ],
        [0.85882353, 0.60392157, 0.2745098 ]],

       [[0.17254902, 0.03921569, 0.00392157],
        [0.17647059, 0.03529412, 0.00392157],
        [0.18823529, 0.03137255, 0.00784314],
        ...,
        [0.94509804, 0.73333333, 0.40784314],
        [0.95294118, 0.74117647, 0.40392157],
        [0.94117647, 0.74117647, 0.41960784]],

       ...,

       [[0.09803922, 0.0745098 , 0.0627451 ],
        [0.16470588, 0.09803922, 0.08235294],
        [0.25098039, 0

In [ ]:
model.save('my_model.keras')

In [ ]:
import os
print(os.getcwd())

c:\Users\GASCO\Desktop
